## Cell 1 – Basic imports & device check

In [1]:
import os
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("Running on CPU")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

Using device: cuda
GPU name: NVIDIA GeForce RTX 3060 Ti
CUDA version: 12.1


## Cell 2 – Hard-coded paths (no config file)

In [2]:
# Hard-coded paths (change only if wrong)
BASE = Path("F:/projects/hirdl/FedSSL_Paper/data_set/kits_2d_splitted")

TRAIN_IMG = BASE / "train/images"
TRAIN_MASK = BASE / "train/masks"
VAL_IMG   = BASE / "val/images"
VAL_MASK   = BASE / "val/masks"
TEST_IMG  = BASE / "test/images"
TEST_MASK = BASE / "test/masks"

# Quick check
for d in [TRAIN_IMG, TRAIN_MASK, VAL_IMG, VAL_MASK, TEST_IMG, TEST_MASK]:
    print(f"{d}: exists = {d.exists()}, png files = {len(list(d.glob('*.png')))}")

F:\projects\hirdl\FedSSL_Paper\data_set\kits_2d_splitted\train\images: exists = True, png files = 11860
F:\projects\hirdl\FedSSL_Paper\data_set\kits_2d_splitted\train\masks: exists = True, png files = 11860
F:\projects\hirdl\FedSSL_Paper\data_set\kits_2d_splitted\val\images: exists = True, png files = 2309
F:\projects\hirdl\FedSSL_Paper\data_set\kits_2d_splitted\val\masks: exists = True, png files = 2309
F:\projects\hirdl\FedSSL_Paper\data_set\kits_2d_splitted\test\images: exists = True, png files = 3808
F:\projects\hirdl\FedSSL_Paper\data_set\kits_2d_splitted\test\masks: exists = True, png files = 3808


## Cell 3 – Minimal Dataset class

In [3]:
class SimpleRCCDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None, min_tumor_px=800):
        self.img_dir = Path(img_dir)
        self.mask_dir = Path(mask_dir)
        self.transform = transform
        self.min_tumor_px = min_tumor_px
        
        self.img_paths = sorted(list(self.img_dir.glob("*.png")))
        print(f"Dataset size: {len(self.img_paths)} slices")
    
    def __len__(self):
        return len(self.img_paths)
    
    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        mask_path = self.mask_dir / img_path.name
        
        # Load image as grayscale → convert to 3 channels for ResNet
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Failed to load image: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        
        # Load mask
        mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
        if mask is None:
            raise ValueError(f"Failed to load mask: {mask_path}")
        
        # Binary label: tumor present if enough 255 pixels
        tumor_count = np.sum(mask == 255)
        label = 1 if tumor_count >= self.min_tumor_px else 0
        
        if self.transform:
            img = self.transform(img)
        
        return img, label

## Cell 4 – Transforms & DataLoaders

In [4]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3),  # simple zero-center
])

# Datasets
train_dataset = SimpleRCCDataset(TRAIN_IMG, TRAIN_MASK, transform=transform, min_tumor_px=800)
val_dataset   = SimpleRCCDataset(VAL_IMG,   VAL_MASK,   transform=transform, min_tumor_px=800)

# Loaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print("Loaders created successfully")

Dataset size: 11860 slices
Dataset size: 2309 slices
Loaders created successfully


## Cell 5 – Quick visualization of a batch

In [ ]:
import tqdm
def show_batch(loader, n=6):
    imgs, labels = next(iter(loader))
    imgs = imgs[:n]
    labels = labels[:n]
    
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.ravel()
    for tqdm in range(n):
        for i in range(n):
            img = imgs[i].permute(1,2,0).numpy() * 0.5 + 0.5
            img = np.clip(img, 0, 1)
            axes[i].imshow(img)
            axes[i].set_title(f"Label: {labels[i].item()}")
            axes[i].axis('off')
        plt.tight_layout()
        plt.show()

show_batch(train_loader)

## Cell 6 – Quick label balance check (on train set)

In [ ]:
labels_list = []
for _, label in tqdm(train_dataset, total=len(train_dataset), desc="Counting labels"):
    labels_list.append(label)

unique, counts = np.unique(labels_list, return_counts=True)
percent = counts / len(labels_list) * 100
print(f"Class distribution (min_tumor_px=800):")
for u, c, p in zip(unique, counts, percent):
    print(f"Label {u}: {c} samples ({p:.2f}%)")